# 01 - Frozen + handcrafted baselines (5-fold CV)

Uses the CORRECTED frozen features from notebook 00 (`results_withseed/frozen_tiny.npy`,
`frozen_base.npy`) and the handcrafted 6-metric features. All on the same seed-123 folds.

- LR baselines (accuracy + AUC), the headline numbers.
- A 7-classifier robustness panel: is LR representative, or does another algorithm win?

Deterministic (fixed folds, convex/ seeded classifiers), so it reproduces exactly on re-run.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
DATA='../data'; OUT='../results_withseed'
labels = np.load(f'{DATA}/preprocessed/labels.npy')
sets = {
    'Frozen tiny': np.load(f'{OUT}/frozen_tiny.npy'),
    'Frozen base': np.load(f'{OUT}/frozen_base.npy'),
    'Handcrafted': np.load(f'{DATA}/baseline_features.npy'),
}
cv = StratifiedKFold(5, shuffle=True, random_state=123)
print({k: v.shape for k, v in sets.items()})

{'Frozen tiny': (148, 4096), 'Frozen base': (148, 8192), 'Handcrafted': (148, 6)}


In [2]:
# Headline LR baselines (accuracy + AUC), same folds as everything else
from sklearn.linear_model import LogisticRegression
print(f'{"feature set":14s} {"acc":>6} {"AUC":>7}')
lr_base = {}
for name, X in sets.items():
    r = cross_validate(LogisticRegression(max_iter=2000), X, labels, cv=cv, scoring=['accuracy','roc_auc'])
    a, u = r['test_accuracy'].mean(), r['test_roc_auc'].mean()
    lr_base[name] = (a, u)
    print(f'{name:14s} {a:.3f}  {u:.3f}')
print('\nCross-check: old (buggy) numbers were Frozen 0.831 / Handcrafted 0.872.')

feature set       acc     AUC
Frozen tiny    0.831  0.859
Frozen base    0.831  0.876
Handcrafted    0.872  0.931

Cross-check: old (buggy) numbers were Frozen 0.831 / Handcrafted 0.872.


In [3]:
# Robustness panel: 7 classifiers x 3 feature sets, same folds, all seeded. Report EVERY cell.
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
S = 123
def sc(clf): return make_pipeline(StandardScaler(), clf)
clfs = {
    'LR (L2)':       LogisticRegression(max_iter=2000),          # no scaling (scaling hurts frozen)
    'Linear SVM':    sc(LinearSVC(max_iter=5000, random_state=S)),
    'RBF SVM':       sc(SVC(kernel='rbf', random_state=S)),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=S, n_jobs=-1),
    'HistGradBoost': HistGradientBoostingClassifier(random_state=S),
    'kNN (k=7)':     sc(KNeighborsClassifier(n_neighbors=7)),
    'MLP (64)':      sc(MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=S)),
}
print(f'{"":15s} ' + ' '.join(f'{n:>13s}' for n in sets) + '   (acc / AUC)')
for cname, clf in clfs.items():
    cells = []
    for X in sets.values():
        r = cross_validate(clf, X, labels, cv=cv, scoring=['accuracy','roc_auc'])
        cells.append(f'{r["test_accuracy"].mean():.3f}/{r["test_roc_auc"].mean():.3f}')
    print(f'{cname:15s} ' + ' '.join(f'{c:>13s}' for c in cells), flush=True)
print('\nExpect LR to be the best (or tied-best) column-wise; nonlinear methods should not beat it.')

                  Frozen tiny   Frozen base   Handcrafted   (acc / AUC)
LR (L2)           0.831/0.859   0.831/0.876   0.872/0.931
Linear SVM        0.784/0.835   0.776/0.847   0.858/0.929
RBF SVM           0.817/0.866   0.803/0.882   0.851/0.908
Random Forest     0.831/0.864   0.804/0.865   0.831/0.894
HistGradBoost     0.844/0.874   0.804/0.856   0.797/0.859
kNN (k=7)         0.811/0.836   0.790/0.841   0.845/0.897
MLP (64)          0.797/0.833   0.811/0.856   0.817/0.895

Expect LR to be the best (or tied-best) column-wise; nonlinear methods should not beat it.
